# Predictive local learning on idealized ring tuning curves

**Julian Hoyer's experiment**, added to a fork of Clark, Abbott and Sompolinsky's
*Symmetries and continuous attractors in disordered neural circuits* repository.
This notebook implements the requested Urbanczik–Senn / Vafidis-style **rate rule**;
it is an extension, rather than a reproduction of a learning experiment in the original repository.
The model setup follows the **Weight-noise robustness sweep** in
[`weight_matrix_analyses.ipynb`](weight_matrix_analyses.ipynb).

Run the cells in order using the repository's Python environment (NumPy, SciPy,
PyTorch, Matplotlib, pandas, tqdm, Jupyter). No cached data are needed. The original
`ring/` code is imported without modification; nothing is written to `data/`.
All generated model, checkpoint, metric and figure files go to the ignored
`notebooks/predictive_rule_outputs/` directory. The default experiment includes
six independent 30,000-step runs and can take hours on CPU.

For a small end-to-end check, launch Jupyter with `PREDICTIVE_RULE_SMOKE=1`, or use:

```bash
PREDICTIVE_RULE_SMOKE=1 python -m jupyter nbconvert --execute --to notebook \
    --ExecutePreprocessor.timeout=-1 --output-dir=/tmp \
    --output=predictive_rule_smoke.ipynb notebooks/predictive_rule.ipynb
```

Smoke mode changes population size, training length and evaluation duration;
its outputs live in a separate `smoke/` subdirectory and are **not scientific results**.
The committed notebook is intentionally unexecuted; only smoke testing was requested.

In [ ]:
import json
import os
import platform
import sys
import time
from pathlib import Path

from matplotlib.figure import Figure
import numpy as np
import pandas as pd
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import torch
from IPython.display import Image, display
from tqdm.auto import tqdm

# Resolve this checkout before importing ring, including from notebooks/.
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'ring' / 'sim.py').is_file():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'ring' / 'sim.py').is_file():
    raise RuntimeError('Run from the repository root or its notebooks/ directory.')
sys.path.insert(0, str(REPO_ROOT))
from ring import data_driven, fn, sim, tc

SMOKE = os.environ.get('PREDICTIVE_RULE_SMOKE', '0') == '1'
N, N_theta = (64, 40) if SMOKE else (1000, 500)
beta, scale, lamda = 2.7659, 1.4211, 1e-8
tau = 0.050
dt = tau / 10
eta0 = 0.5
BATCH = 50
STEPS = 12 if SMOKE else 30_000
CHECKPOINTS = [2, 6, 12] if SMOKE else [10, 30, 100, 300, 1000, 3000, 10000, 30000]
LAMS = (1.0, 0.5)
NOISE_LEVELS = (0.0, 0.1, 0.3)
N_INIT = 6 if SMOKE else 34
T_TOTAL = 1.1 if SMOKE else 30.0
T_EVAL = 0.05
F_INIT = 0.025
ANGLE_TOL_DEG = 3.0
SEED = 123

# Float64 makes the 1e-8 ridge reference and fixed-point checks reproducible
# at high precision. CUDA is used when available; otherwise use CPU.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.float64
torch.set_num_threads(min(4, os.cpu_count() or 1))
SOMA_ATOL, SOMA_RTOL = 1e-8, 1e-6
SOMA_MAX_ITERS = 2000
SOMA_RELAXATION = 1.0  # Picard iteration; lower this if convergence fails.

OUTPUT_DIR = REPO_ROOT / 'notebooks' / 'predictive_rule_outputs' / ('smoke' if SMOKE else 'full')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)  # tc.sample_tcs uses NumPy's global RNG.
torch.manual_seed(SEED)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True)

def nonlin(x):
    return fn.nonlin_idealized(x, beta)

def as_tensor(x):
    return torch.as_tensor(x, dtype=DTYPE, device=DEVICE)

def save_json(path, obj):
    # Strict JSON: NaN scientific metrics are saved separately in NPZ files.
    path.write_text(json.dumps(obj, indent=2, allow_nan=False) + '\n')

def show_figure(fig, filename):
    # Render explicitly, also supporting older Matplotlib with newer IPython.
    path = OUTPUT_DIR / filename
    fig.savefig(path, dpi=160)
    display(Image(filename=str(path)))

config = dict(
    smoke=SMOKE, N=N, N_theta=N_theta, beta=beta, scale=scale, lamda=lamda,
    tau=tau, dt=dt, eta0=eta0, batch=BATCH, steps=STEPS,
    checkpoints=CHECKPOINTS, lams=LAMS, noise_levels=NOISE_LEVELS,
    n_init=N_INIT, T_total=T_TOTAL, T_eval=T_EVAL, initial_noise=F_INIT,
    angle_tolerance_deg=ANGLE_TOL_DEG, seed=SEED, dtype=str(DTYPE), device=str(DEVICE),
    soma_atol=SOMA_ATOL, soma_rtol=SOMA_RTOL, soma_max_iters=SOMA_MAX_ITERS,
    soma_relaxation=SOMA_RELAXATION,
    update_noise='iid Gaussian; entry std = relative_noise * RMS(clean off-diagonal update)',
    python=platform.python_version(), numpy=np.__version__, torch=torch.__version__,
)
config_path = OUTPUT_DIR / 'config.json'
# Compare JSON-normalized types (tuples become lists) before any model writes.
if config_path.exists() and json.loads(config_path.read_text()) != json.loads(json.dumps(config)):
    raise RuntimeError(f'Configuration changed. Move {OUTPUT_DIR} aside before rerunning.')
save_json(config_path, config)
print(('SMOKE CHECK' if SMOKE else 'FULL EXPERIMENT'), '|', DEVICE, '|', OUTPUT_DIR)

## Tuning curves, references and matched initial states

Rows index angles, columns index neurons: $H_{\theta i}=h_i(\theta)$ and
$\Phi=\phi(H)$. The two reference matrices use exactly the requested formulas:

\[
J_{\rm paper}=\operatorname{compute\_weights}(H,\Phi,10^{-8})_0,\qquad
J_{\rm cov}=\frac{g_1}{N N_\theta}(H-\overline H)^T(\Phi-\overline\Phi),
\quad g_1=1/\widehat C_\Phi(1).
\]

Only the covariance reference's diagonal is explicitly zeroed here; the paper
reference is used exactly as returned by `compute_weights`. All evaluations use
the **same** 34 evenly spaced input states and the **same** additive Gaussian
noise, whose standard deviation is 2.5% of each initial state's neuron-wise
standard deviation, as in the original sweep. There is no drive during evaluation.

In [ ]:
H, Phi = tc.gen_tuning_curves_idealized(N, N_theta, scale, beta)
assert H.shape == Phi.shape == (N_theta, N)
np.testing.assert_allclose(Phi, nonlin(H))
J_paper, _ = data_driven.compute_weights(H, Phi, lamda)
g1 = 1.0 / fn.calc_C(Phi.T)[1][1]
J_cov = g1 / (N * N_theta) * ((H - H.mean(0)).T @ (Phi - Phi.mean(0)))
np.fill_diagonal(J_cov, 0.0)

theta = 2 * np.pi * np.arange(N_theta) / N_theta
theta_idx = np.rint(np.arange(N_INIT) * N_theta / N_INIT).astype(int) % N_theta
start_deg = np.rad2deg(theta[theta_idx])
initial_rng = np.random.default_rng(SEED + 1)
x_init = H[theta_idx].T.copy()
x_init += F_INIT * x_init.std(axis=0, keepdims=True) * initial_rng.standard_normal(x_init.shape)
H_t, Phi_t = as_tensor(H), as_tensor(Phi)
J_paper_t = as_tensor(J_paper)
paper_norm = torch.linalg.vector_norm(J_paper_t)

# Exact first-harmonic projection of m(theta) = Phi @ phi(x) / N.
# The positive exponential gives the angle itself (rfft would give its negative).
harmonic_projection = np.exp(1j * theta) @ Phi / (N * N_theta)
dc_projection = Phi.mean(axis=0) / N
np.savez_compressed(
    OUTPUT_DIR / 'model.npz', H=H, Phi=Phi, J_paper=J_paper, J_cov=J_cov,
    x_init=x_init, theta=theta, theta_idx=theta_idx, g1=g1,
)
print(f'g1 = {g1:.6g}; zero-input evaluation duration = {T_TOTAL:g} s')

## Steady-state soma and the local update

Each training step samples 50 angles **with replacement**. Every run starts at $J=0$.
For $\lambda=1$ the soma is set to $x=h$ exactly. For $\lambda=0.5$, Picard
iteration starts at the current minibatch's $h$ and solves

\[
x=(1-\lambda)J\phi(x)+\lambda h,\quad p=J\phi(x),\quad
\Delta J=\frac{\eta_0}{N B}(x-p)\phi(x)^T.
\]

The per-example RMS fixed-point residual must be below
`SOMA_ATOL + SOMA_RTOL * RMS(h)`; failure stops the run with a diagnostic, rather
than treating a transient as a steady state. `SOMA_RELAXATION` can be lowered if
needed. No centering, ridge penalty, symmetrization, weight clipping, or extra
normalization is applied to this update. The diagonal is zero after every step.

For the robustness runs, noise is added **at every step** to the clean update,
before applying the zero-diagonal constraint. Off-diagonal noise entries are iid
Gaussian with standard deviation $f\,\operatorname{RMS}(\Delta J_{i\ne j})$,
for $f=0.1,0.3$. Thus these values describe per-synapse update noise, rather than
post-training weight perturbations. All runs share the same minibatch sequence;
update noise has a separate, matched random stream. All six runs are independent
and start from zero, including the repeated $\lambda=0.5$ runs.

In [ ]:
@torch.no_grad()
def solve_soma(J, h, lam):
    if lam == 1.0:
        return h.clone(), 0, 0.0
    x = h.clone()
    tolerance = SOMA_ATOL + SOMA_RTOL * h.square().mean(dim=0).sqrt()
    for iteration in range(SOMA_MAX_ITERS + 1):
        residual = (1 - lam) * (J @ nonlin(x)) + lam * h - x
        scaled_residual = residual.square().mean(dim=0).sqrt() / tolerance
        ratio = scaled_residual.max().item()
        if not np.isfinite(ratio):
            raise RuntimeError('Non-finite somatic current; no update was applied.')
        if ratio <= 1:
            return x, iteration, ratio
        x.add_(residual, alpha=SOMA_RELAXATION)
    raise RuntimeError(
        f'Soma failed to converge at lambda={lam}: residual/tolerance={ratio:.3g}. '
        'Try a smaller SOMA_RELAXATION; record the new configuration and rerun.'
    )

@torch.no_grad()
def predictive_update(J, h, lam, relative_noise, noise_generator):
    x, iterations, residual_ratio = solve_soma(J, h, lam)
    rates = nonlin(x)
    p = J @ rates
    update = (eta0 / N) * ((x - p) @ rates.T) / h.shape[1]
    update.fill_diagonal_(0)
    if relative_noise:
        offdiag_rms = update.square().sum().div(N * (N - 1)).sqrt()
        noise = torch.randn(J.shape, generator=noise_generator, device=DEVICE, dtype=DTYPE)
        update.add_(noise * (relative_noise * offdiag_rms))
    J.add_(update)
    J.fill_diagonal_(0)
    if not torch.isfinite(J).all():
        raise RuntimeError('Non-finite learned weights; stopping this run.')
    return iterations, residual_ratio

## Input-off measurements

For every clean-run checkpoint (and the last checkpoint of every noisy run):

* **Flow error:** $\sqrt{\langle(-H+\Phi J^T)^2\rangle_{\theta,i}}$.
* **Decoded angle:** $\arg[\sum_\theta m(\theta)e^{i\theta}]$, where
  $m=\Phi\phi(x)/N$. The projection is precomputed without changing the decoder.
  Phase is unwrapped in time. Drift speed is the mean absolute angular speed
  across saved 0.05-second intervals in **[0.5, 1] s**, then averaged across starts,
  in degrees/second. Per-start speeds and signed net drifts are also saved.
* **Distinct final angles:** minimum number of circular clusters covering the
  final decoded angles, each with angular diameter at most 3°. Trying every
  circular cut handles the 0°/360° seam without joining a chain of distant angles.
* **Weight error:** $\|J-J_{\rm paper}\|_F/\|J_{\rm paper}\|_F$.
* **Spectrum:** top 12 singular values (exact SVD).

`ring.sim.run_sim` is called with input off, no stabilizer, and early stopping
disabled. That integrator's loop ends at `T - dt`; a second plain `run_sim` call
takes the one remaining Euler step, so the final angle is measured at exactly
30 s (1.1 s in smoke mode). The full saved trajectory is decoded before being
discarded; weights, angles, harmonic strengths and metrics remain on disk.
Undefined phases (relative harmonic strength below $10^{-10}$) become NaN and are
excluded from drift averages; if any final phase is undefined, the cluster count
is NaN. Harmonic strength and the number of valid starts are reported so that a
decoded angle alone is not mistaken for evidence of a surviving ring attractor.

In [ ]:
def decode_states(states):
    """Input (..., neurons, starts); return angle and relative first harmonic."""
    rates = nonlin(np.asarray(states))
    z = np.einsum('n,...nb->...b', harmonic_projection, rates)
    dc = np.einsum('n,...nb->...b', dc_projection, rates)
    strength = np.abs(z) / np.maximum(np.abs(dc), np.finfo(float).tiny)
    angle = np.angle(z)
    angle[(strength < 1e-10) | ~np.isfinite(strength)] = np.nan
    return angle, strength

def count_final_angles(angles, tolerance_deg=ANGLE_TOL_DEG):
    angles = np.asarray(angles)
    if not np.isfinite(angles).all():
        return np.nan
    values = np.sort(np.rad2deg(angles) % 360)
    if not len(values):
        return 0
    # A valid cluster is a circular arc of length <= tolerance; no chaining.
    extended = np.r_[values, values + 360]
    best = len(values)
    for cut in range(len(values)):
        clusters, end = 0, -np.inf
        for value in extended[cut:cut + len(values)]:
            if value > end + 1e-9:
                clusters += 1
                end = value + tolerance_deg
        best = min(best, clusters)
    return best

@torch.no_grad()
def measure(J, label, directory):
    directory.mkdir(parents=True, exist_ok=True)
    J_t = as_tensor(J)
    started = time.perf_counter()
    final_x, history, finished = sim.run_sim(
        T=T_TOTAL, tau=tau, dt=dt, T_eval=T_EVAL,
        J=J_t, x_init=as_tensor(x_init.copy()), nonlin=nonlin,
        break_if_stopped=False, disable_tqdm=True,
    )
    # run_sim performs int(T/dt)-1 updates: complete exactly T/dt updates.
    target_steps = int(round(T_TOTAL / dt))
    completed_steps = int(T_TOTAL / dt) - 1
    remaining = target_steps - completed_steps
    final_x, _, _ = sim.run_sim(
        T=(remaining + 1) * dt, tau=tau, dt=dt, T_eval=None,
        J=J_t, x_init=as_tensor(final_x.copy()), nonlin=nonlin,
        break_if_stopped=False, disable_tqdm=True,
    )
    times = np.arange(history.shape[0]) * T_EVAL
    # Decode in chunks to avoid allocating a second complete rate history.
    decoded = [decode_states(history[i:i + 16]) for i in range(0, len(history), 16)]
    angles = np.concatenate([item[0] for item in decoded])
    strength = np.concatenate([item[1] for item in decoded])
    final_angles, final_strength = decode_states(final_x)
    window = (times >= 0.5 - 1e-9) & (times <= 1.0 + 1e-9)
    window_times = times[window]
    assert len(window_times) >= 2
    assert np.isclose(window_times[0], 0.5) and np.isclose(window_times[-1], 1.0)
    window_angles = angles[window]
    valid_drift = np.isfinite(window_angles).all(axis=0)
    speeds = np.full(N_INIT, np.nan)
    net_drift = np.full(N_INIT, np.nan)
    if valid_drift.any():
        unwrapped = np.unwrap(window_angles[:, valid_drift], axis=0)
        increments = np.rad2deg(np.diff(unwrapped, axis=0))
        speeds[valid_drift] = np.abs(increments).sum(axis=0) / (window_times[-1] - window_times[0])
        net_drift[valid_drift] = np.rad2deg(unwrapped[-1] - unwrapped[0]) / (window_times[-1] - window_times[0])
    result = dict(
        label=label,
        flow_error=float(((-H_t + Phi_t @ J_t.T).square().mean().sqrt()).cpu()),
        drift_deg_s=float(np.mean(speeds[valid_drift])) if valid_drift.any() else np.nan,
        distinct_final=count_final_angles(final_angles),
        relative_weight_error=float((torch.linalg.vector_norm(J_t - J_paper_t) / paper_norm).cpu()),
        singular_values=torch.linalg.svdvals(J_t)[:12].cpu().numpy(),
        start_deg=start_deg.copy(), initial_angles=angles[0], final_angles=final_angles,
        drift_per_start_deg_s=speeds, net_drift_per_start_deg_s=net_drift,
        decoded_times=times, decoded_angles=angles, harmonic_strength=strength,
        final_harmonic_strength=final_strength, final_state=final_x,
        valid_drift_starts=int(valid_drift.sum()), valid_final_starts=int(np.isfinite(final_angles).sum()),
        evaluation_seconds=time.perf_counter() - started,
        final_time_s=target_steps * dt, simulation_finished=bool(finished),
    )
    np.savez_compressed(directory / 'metrics.npz', **result)
    return result

## Measure the two reference networks

The horizontal reference lines use the same decoder, input-off dynamics,
noise realization and measurement times as the learned matrices.

In [ ]:
references = {}
for label, weights in [('J_paper', J_paper), ('J_cov', J_cov)]:
    references[label] = measure(weights, label, OUTPUT_DIR / label)
display(pd.DataFrame([
    {key: result[key] for key in (
        'label', 'flow_error', 'drift_deg_s', 'distinct_final',
        'relative_weight_error', 'valid_drift_starts', 'valid_final_starts',
    )} for result in references.values()
]))

## Train and save log-spaced checkpoints

Clean runs are evaluated at every checkpoint. Noisy runs save weights at the
same checkpoints and evaluate input-off behavior at the final one. Each checkpoint
contains the full `J`, step, coupling, noise level and solver diagnostics. Files are
overwritten when rerunning the same configuration; copy an output directory first
if you want to preserve a previous experiment. A configuration mismatch stops
before overwriting any model or checkpoint files.

In [ ]:
@torch.no_grad()
def train(lam, relative_noise):
    run_name = f'lam_{lam:g}_noise_{relative_noise:g}'
    directory = OUTPUT_DIR / run_name
    directory.mkdir(parents=True, exist_ok=True)
    J = torch.zeros((N, N), dtype=DTYPE, device=DEVICE)
    angle_rng = torch.Generator(device=DEVICE).manual_seed(SEED + 2)
    noise_rng = torch.Generator(device=DEVICE).manual_seed(SEED + 3)
    results, diagnostics = {}, []
    started = time.perf_counter()
    iterations_sum, iterations_max, residual_max = 0, 0, 0.0
    checkpoint_set = set(CHECKPOINTS)
    last_step = 0
    for step in tqdm(range(1, STEPS + 1), desc=run_name):
        indices = torch.randint(N_theta, (BATCH,), device=DEVICE, generator=angle_rng)
        h = H_t[indices].T
        iterations, residual = predictive_update(J, h, lam, relative_noise, noise_rng)
        iterations_sum += iterations
        iterations_max = max(iterations_max, iterations)
        residual_max = max(residual_max, residual)
        if step in checkpoint_set:
            checkpoint = directory / f'step_{step:05d}'
            checkpoint.mkdir(parents=True, exist_ok=True)
            diagnostic = dict(
                step=step, mean_soma_iterations=iterations_sum / (step - last_step),
                max_soma_iterations=iterations_max, max_residual_over_tolerance=residual_max,
                elapsed_seconds=time.perf_counter() - started,
            )
            np.savez_compressed(
                checkpoint / 'weights.npz', J=J.cpu().numpy(), step=step,
                lam=lam, relative_noise=relative_noise, **{k: v for k, v in diagnostic.items() if k != 'step'},
            )
            diagnostics.append(diagnostic)
            save_json(directory / 'training_diagnostics.json', diagnostics)
            if relative_noise == 0 or step == STEPS:
                results[step] = measure(J, f'{run_name}, step={step}', checkpoint)
            last_step = step
            iterations_sum, iterations_max, residual_max = 0, 0, 0.0
    return results

runs = {}
# First the complete clean lambda=1 run, then a fresh clean lambda=0.5 run.
for relative_noise in NOISE_LEVELS:
    for lam in LAMS:
        runs[(lam, relative_noise)] = train(lam, relative_noise)

## Learning curves and angle maps

Training steps are logarithmic. The drift and weight-error axes use a symmetric
log scale near zero to retain the exact $J_{\rm paper}$ distance reference of zero;
the cluster-count axis is logarithmic. Dashed horizontal lines are measured
reference networks. The angle maps show the first, middle and last checkpoints
of each clean run, against the nominal input angle used to initialize the state.

In [ ]:
fig = Figure(figsize=(15, 4), layout='constrained')
axes = fig.subplots(1, 3)
metrics = [
    ('drift_deg_s', 'Drift speed [deg/s]'),
    ('distinct_final', f'Distinct final angles ({ANGLE_TOL_DEG:g}° tolerance)'),
    ('relative_weight_error', r'$\|J-J_{paper}\|_F / \|J_{paper}\|_F$'),
]
for ax, (metric, ylabel) in zip(axes, metrics):
    for lam in LAMS:
        clean = runs[(lam, 0.0)]
        ax.plot(CHECKPOINTS, [clean[s][metric] for s in CHECKPOINTS], 'o-', label=fr'$\lambda={lam:g}$')
    for label, color in [('J_paper', 'black'), ('J_cov', 'gray')]:
        ax.axhline(references[label][metric], ls='--', color=color, label=label)
    ax.set_xscale('log')
    if metric == 'distinct_final':
        ax.set_yscale('log')
    else:
        ax.set_yscale('symlog', linthresh=1e-6 if metric == 'drift_deg_s' else 1e-4)
        ax.set_ylim(bottom=0)
    ax.set(xlabel='Training step', ylabel=ylabel)
    ax.grid(alpha=0.25)
axes[0].legend()
show_figure(fig, 'learning_curves.png')

selected = [CHECKPOINTS[0], CHECKPOINTS[len(CHECKPOINTS) // 2], CHECKPOINTS[-1]]
fig = Figure(figsize=(12, 8), layout='constrained')
axes = fig.subplots(len(LAMS), 3, squeeze=False)
for row, lam in enumerate(LAMS):
    for col, step in enumerate(selected):
        result = runs[(lam, 0.0)][step]
        ax = axes[row, col]
        ax.scatter(start_deg, np.rad2deg(result['final_angles']) % 360)
        ax.plot([0, 360], [0, 360], 'k--', alpha=0.4)
        ax.set(xlim=(0, 360), ylim=(0, 360), xlabel='Start angle [deg]',
               ylabel=f'Final angle at {T_TOTAL:g} s [deg]', title=fr'$\lambda={lam:g}$, step {step}')
        ax.set_aspect('equal')
show_figure(fig, 'start_final_angles.png')

## Flow errors and singular spectra at every clean checkpoint

Each spectrum is shown beside the paper reference; the covariance reference is
included for comparison. Full metric arrays, including individual decoded
trajectories, are saved in each checkpoint's `metrics.npz` without pickle objects.

In [ ]:
checkpoint_rows = []
for lam in LAMS:
    for step in CHECKPOINTS:
        result = runs[(lam, 0.0)][step]
        checkpoint_rows.append(dict(
            lam=lam, step=step,
            **{key: result[key] for key in (
                'flow_error', 'drift_deg_s', 'distinct_final', 'relative_weight_error',
                'valid_drift_starts', 'valid_final_starts',
            )},
        ))
checkpoint_table = pd.DataFrame(checkpoint_rows)
display(checkpoint_table)
checkpoint_table.to_csv(OUTPUT_DIR / 'clean_checkpoint_metrics.csv', index=False)

fig = Figure(figsize=(3 * len(CHECKPOINTS), 6), layout='constrained')
axes = fig.subplots(len(LAMS), len(CHECKPOINTS), squeeze=False)
for row, lam in enumerate(LAMS):
    for col, step in enumerate(CHECKPOINTS):
        ax = axes[row, col]
        for label, result, style in [
            ('learned', runs[(lam, 0.0)][step], 'o-'),
            ('J_paper', references['J_paper'], 'k--'),
            ('J_cov', references['J_cov'], ':')]:
            values = result['singular_values']
            ax.plot(np.arange(1, len(values) + 1), values, style, label=label)
        ax.set(title=fr'$\lambda={lam:g}$, step {step}', xlabel='Singular-value rank')
        ax.set_yscale('log')
        ax.grid(alpha=0.25)
axes[0, 0].legend()
axes[0, 0].set_ylabel('Singular value')
axes[1, 0].set_ylabel('Singular value')
show_figure(fig, 'singular_spectra.png')

## Robustness to noise added to every learning update

The table reports the final checkpoint for all six runs, with the clean runs
and both fixed references alongside the 0.1 and 0.3 update-noise conditions.
These are single-seed comparisons, not confidence intervals across realizations.

In [ ]:
robustness_rows = []
for (lam, relative_noise), checkpoints in runs.items():
    result = checkpoints[STEPS]
    robustness_rows.append(dict(
        network='learned', lam=lam, update_noise=relative_noise, step=STEPS,
        drift_deg_s=result['drift_deg_s'], distinct_final=result['distinct_final'],
        valid_drift_starts=result['valid_drift_starts'], valid_final_starts=result['valid_final_starts'],
        median_final_harmonic_strength=float(np.median(result['final_harmonic_strength'])),
    ))
for label, result in references.items():
    robustness_rows.append(dict(
        network=label, lam=np.nan, update_noise=np.nan, step=np.nan,
        drift_deg_s=result['drift_deg_s'], distinct_final=result['distinct_final'],
        valid_drift_starts=result['valid_drift_starts'], valid_final_starts=result['valid_final_starts'],
        median_final_harmonic_strength=float(np.median(result['final_harmonic_strength'])),
    ))
robustness_table = pd.DataFrame(robustness_rows)
display(robustness_table)
robustness_table.to_csv(OUTPUT_DIR / 'robustness_final_metrics.csv', index=False)

fig = Figure(figsize=(10, 4), layout='constrained')
axes = fig.subplots(1, 2)
for ax, metric, ylabel in zip(axes, ['drift_deg_s', 'distinct_final'],
                            ['Drift speed [deg/s]', 'Distinct final angles']):
    for lam in LAMS:
        ax.plot(NOISE_LEVELS, [runs[(lam, f)][STEPS][metric] for f in NOISE_LEVELS],
                'o-', label=fr'$\lambda={lam:g}$')
    for label, color in [('J_paper', 'black'), ('J_cov', 'gray')]:
        ax.axhline(references[label][metric], color=color, ls='--', label=label)
    ax.set(xlabel='Relative update-noise standard deviation', ylabel=ylabel)
    ax.grid(alpha=0.25)
axes[0].legend()
show_figure(fig, 'update_noise_robustness.png')
print('Saved checkpoints, metrics, tables and figures to', OUTPUT_DIR)